# pob

> the four tables on one vault, and the tick that drives them

In [ ]:
#| default_exp pob

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

One object holding the queue, the lists, the notes and the schedules on a single vault. This is
what ramabana and vishalakshi keep, and what `pob tick` opens.

Pass `db` where the vault is already open. All four tables then share that connection, litesearch's
lock and its transaction. A document write and the job it causes commit together, and roll back
together.

In [ ]:
#| export
from fastcore.all import AttrDict, L, ifnone, patch, store_attr
from pobblebonk.core import BUSY_TIMEOUT_MS, now, vault
from pobblebonk.inbox import Inbox
from pobblebonk.lists import Lists
from pobblebonk.queue import Queue
from pobblebonk.run import attach
from pobblebonk.schedule import FIRE, Schedules

In [ ]:
#| export
class Pob:
    "The queue, the lists, the notes and the schedules on one vault."
    def __init__(self,
                 path=None,                     # the vault file; in memory when None
                 db=None,                       # or a vault something else already opened
                 busy_ms:int=BUSY_TIMEOUT_MS,   # lock wait before a write gives up
                 **kw):                         # passed to `Queue`: lease, max_attempts, base, keep
        self.db = db if db is not None else vault(path, busy_ms)
        self.queue = Queue(self.db, busy_ms=busy_ms, **kw)
        self.lists = Lists(self.db, busy_ms)
        self.inbox = Inbox(self.db, busy_ms)
        self.scheds = Schedules(self.db, busy_ms)

    def __repr__(self):
        s = self.queue.stats()
        return (f'Pob({len(self.scheds.all())} schedules, {s["ready"]} ready, {s["dead"]} dead, '
                f'{self.inbox.count()} notes)')

    def attach(self, runner, cwd:str=None) -> 'Pob':
        'Say what runs a fire. Without this a fire has no handler and dead-letters.'
        attach(self.queue, runner, self.scheds, self.inbox, self.lists, cwd)
        return self

## The tick

Three things in order: give back the leases of workers that died, turn due schedules into jobs,
then run the jobs. A fire that used up its attempts leaves a note of its own, because a reminder
that fails silently is worse than one that never existed.

In [ ]:
#| export
@patch
def tick(self:Pob,
         worker:str='tick',    # the lease holder recorded on each job
         at:float=None,        # when this tick is happening
         limit:int=50,         # schedules looked at
         run:bool=True         # False queues the fires and runs nothing, for a dry tick
) -> AttrDict:
    'Reclaim, fire what is due, then work the queue. This is what a cron entry calls.'
    at = ifnone(at, now())
    back = self.queue.reclaim(at)
    out = self.scheds.tick(self.queue, at=at, inbox=self.inbox, have=self.lists.count, limit=limit)
    ran = self.queue.drain(worker, at=at) if run else L()
    for r in ran:
        j = self.queue.get(r['job_id'])
        if j is not None and j['state'] == 'dead':
            self.inbox.note('dead', f"{self.name_of(j)} gave up after {j['attempt']} tries",
                            j['error'] or '', ref=j['id'], at=at)
    return AttrDict(fired=out.fired, blocked=out.blocked, freed=out.freed, ran=ran, reclaimed=back)

@patch
def name_of(self:Pob, job) -> str:
    "What to call a job in a note: its schedule's name where it has one."
    if job['kind'] == FIRE and (s := self.scheds.get((job['payload'] or {}).get('schedule', ''))):
        return s['name']
    return job['kind']

@patch
def drain(self:Pob, reader:str, limit:int=50) -> L:
    'Every note this reader has not had. What a turn calls before it answers you.'
    return self.inbox.drain(reader, limit)

@patch
def status(self:Pob) -> AttrDict:
    'One line of everything, for `pob status` and for a status bar.'
    s = self.queue.stats()
    return AttrDict(schedules=len(self.scheds.all()), paused=len(self.scheds.all(paused=True)),
                    due=len(self.scheds.due()), notes=self.inbox.count(),
                    topics={t['topic']: t['n'] for t in self.lists.topics()}, **s)

The whole of the Wednesday cart, through one object:

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pobblebonk.run import FakeRunner

kol = ZoneInfo('Asia/Kolkata')
wed = datetime(2026, 8, 26, 20, 1, tzinfo=kol).timestamp()

pob = Pob().attach(FakeRunner(['added 6 items, no oat milk']))
pob.lists.add('shopping', 'oat milk')
pob.scheds.add('cart', 'Fill my cart from the shopping list.', cron='0 20 * * 3',
               tz='Asia/Kolkata', needs='shopping', policy='write', start=wed - 60)

out = pob.tick(at=wed)
test_eq(len(out.fired), 1)
test_eq(out.ran[0].status, 'ok')
test_eq(pob.drain('leela').attrgot('body'), ['added 6 items, no oat milk'])
test_eq(pob.status().topics, {})

A fire that keeps failing says so rather than disappearing:

In [ ]:
p2 = Pob(max_attempts=1, base=0).attach(FakeRunner([RuntimeError('the shop was down')]))
p2.scheds.add('cart', 'Fill my cart.', every='1h', start=0)
p2.tick(at=1)
test_eq(p2.inbox.notes(kind='dead')[0]['title'], 'cart gave up after 1 tries')
test_eq(len(p2.queue.dead()), 1)

A dry tick queues the work and runs none of it, which is what `pob tick --dry` is for:

In [ ]:
p3 = Pob().attach(FakeRunner())
p3.scheds.add('cart', 'Fill my cart.', every='1h', start=0)
out = p3.tick(at=1, run=False)
test_eq(len(out.fired), 1)
test_eq(out.ran, [])
test_eq(p3.queue.pending(FIRE), 1)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()